# CAFPO FF6 Fixed-180 Experiments

This notebook is an isolated experiment copy for the fixed-universe variants:

1. `exp1_ff6_fixed180`: for each test year, form a fixed 180-stock universe at the first month-end of the test year from Fama-French Size x Book-to-Market six portfolios, 30 stocks per cell. The same 180 stock slots are used for CAE input, PPO actions, training, and testing.
2. `exp2_ff6_fixed180_featurefilter`: based on experiment 1, drop features whose training-period cross-sectional missing rate is at least 50% in more than half of the 120 training months.
3. `exp3_ff6_fixed180_featurefilter_action18_longonly_t*`: based on experiment 2, keep CAE input at 180 stocks, but restrict the PPO action space and tradable baseline universe to 18 stocks, 3 from each FF6 cell and a subset of the 180. Run this as a long-only temperature sweep over 0.2, 0.5, and 1.0.
4. Plan A: use the largest data-complete fixed FF6-balanced pool, 1056 stocks, as CAE input; PPO outputs 6 FF6 group weights, and each group return is monthly value-weighted within the group.

Important implementation notes:

- The data source is the all-stock prepanel `cafpo_82_prepanel_allstocks.parquet`.
- Growth/Neutral/Value breakpoints use `lagged_x_bm` by default, matching the no-lookahead lag convention in the prepanel; pass `value_col="bm"` in `build_ff6_split_data` only for a raw same-month B/M diagnostic.
- The old yearly top-200 model-ready panel is intentionally not loaded.
- The 2017 split is expected to fail under the strict rule that every selected stock must have all 120 training months of `ret_1m`, `ret_fwd_1m`, and at least one available raw lagged feature. In this dataset, 2007-01 has no `ret_1m`, so no stock can satisfy that exact 120-month rule for the 2007-2016 training window.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "cafpo_reproduction.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "cafpo"
assert (NOTEBOOK_DIR / "cafpo_reproduction.py").exists(), NOTEBOOK_DIR
if str(NOTEBOOK_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

from cafpo_reproduction import (
    OUTPUT_DIR,
    extract_cae_outputs,
    make_ppo_model,
    performance_summary,
    run_markowitz_baseline,
    save_cae_outputs,
    set_seed,
    train_cae,
)
from cafpo_ff6_experiments import (
    FF6_GROUP_ORDER,
    FixedActionPortfolioEnv,
    build_ff6_split_data,
    evaluate_raw_action_ensemble,
    run_universe_weight_baseline,
    load_prepanel_and_manifest,
    rolling_test_years,
    subset_arrays,
)

set_seed(42)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_OUTPUT_DIR = OUTPUT_DIR / "ff6_fixed180_experiments"
EXPERIMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_OUTPUT_DIR

WindowsPath('C:/xmrs_workspace/workspace/【广发】深度学习play/cafpo/rqdata_output/ff6_fixed180_experiments')

In [2]:
N_STOCKS = 180
N_FACTORS = 5
LOOKBACK = 12
TOP_BOTTOM_FRAC = 0.30
LONG_ONLY_TEMPERATURE = 0.2
EXP3_LONG_ONLY_TEMPERATURES = [0.2, 0.5, 1.0]

CAE_EPOCHS = 200
CAE_PATIENCE = 20
PPO_TOTAL_TIMESTEPS = 20_000
PPO_N_STEPS = 64
PPO_BATCH_SIZE = 32
DRL_SEEDS = [101, 202, 303, 404, 505]

# Set to True when you are ready to run the full model training loop.
# The full loop is expensive: 5 long-only experiments x feasible rolling years x CAE + 5 PPO seeds.
RUN_FULL_EXPERIMENTS = False

# None means all feasible rolling years. Use a short list, e.g. [2018], for a smoke run.
TEST_YEARS_TO_RUN = None
AGGREGATE_OUTPUT_PREFIX = "all_longonly_experiments"

EXPERIMENTS = [
    {
        "name": "exp1_ff6_fixed180_longonly",
        "label": "Experiment 1: FF6 fixed 180, CAFPO long-only",
        "drop_sparse_features": False,
        "n_action_stocks": 180,
        "action_mode": "long_only",
        "long_only_temperature": LONG_ONLY_TEMPERATURE,
        "cafpo_method": "CAFPO_PPO_LogReturn_LongOnly_5SeedAvgAction",
        "include_markowitz": False,
    },
    {
        "name": "exp2_ff6_fixed180_featurefilter_longonly",
        "label": "Experiment 2: FF6 fixed 180 + feature filter, CAFPO long-only",
        "drop_sparse_features": True,
        "n_action_stocks": 180,
        "action_mode": "long_only",
        "long_only_temperature": LONG_ONLY_TEMPERATURE,
        "cafpo_method": "CAFPO_PPO_LogReturn_LongOnly_5SeedAvgAction",
        "include_markowitz": False,
    },
    *[
        {
            "name": f"exp3_ff6_fixed180_featurefilter_action18_longonly_t{str(temp).replace('.', 'p')}",
            "label": f"Experiment 3: FF6 fixed 180 + feature filter + 18 actions, CAFPO long-only, T={temp}",
            "drop_sparse_features": True,
            "n_action_stocks": 18,
            "action_mode": "long_only",
            "long_only_temperature": temp,
            "cafpo_method": f"CAFPO_PPO_LogReturn_LongOnly_T{temp}_5SeedAvgAction",
            "include_markowitz": False,
        }
        for temp in EXP3_LONG_ONLY_TEMPERATURES
    ],
]

# The previously-run long-short configs are intentionally not the default anymore.
# Their result files remain under ff6_fixed180_experiments for comparison.
LONG_SHORT_EXPERIMENTS = [
    {"name": "exp1_ff6_fixed180", "drop_sparse_features": False, "n_action_stocks": 180, "action_mode": "long_short"},
    {"name": "exp2_ff6_fixed180_featurefilter", "drop_sparse_features": True, "n_action_stocks": 180, "action_mode": "long_short"},
    {"name": "exp3_ff6_fixed180_featurefilter_action18", "drop_sparse_features": True, "n_action_stocks": 18, "action_mode": "long_short"},
]


In [3]:
pre, manifest = load_prepanel_and_manifest()
raw_feature_cols = manifest["lagged_raw_col"].tolist()
safe_feature_cols = manifest["feature_safe"].tolist()

audit = {
    "rows": len(pre),
    "date_start": pre["date"].min(),
    "date_end": pre["date"].max(),
    "n_stocks_total": pre["order_book_id"].nunique(),
    "n_features_manifest": len(manifest),
    "source": "cafpo_82_prepanel_allstocks.parquet",
}
display(pd.Series(audit))
assert len(safe_feature_cols) >= 82
assert {"date", "order_book_id", "ret_1m", "ret_fwd_1m", "feature_nonnull_raw", "mvel1", "lagged_x_bm"}.issubset(pre.columns)

rows                                                754182
date_start                             2007-01-31 00:00:00
date_end                               2026-05-29 00:00:00
n_stocks_total                                        5485
n_features_manifest                                     83
source                 cafpo_82_prepanel_allstocks.parquet
dtype: object

In [4]:
def diagnose_pool_builds(test_years=None):
    if test_years is None:
        test_years = rolling_test_years(pre)
    rows = []
    failures = []
    for year in test_years:
        try:
            data = build_ff6_split_data(
                pre,
                manifest,
                test_year=year,
                n_stocks=N_STOCKS,
                n_action_stocks=18,
                drop_sparse_features=True,
                seed=42,
            )
            row = dict(data.audit)
            row["status"] = "ok"
            rows.append(row)
        except Exception as exc:
            failures.append({"test_year": year, "status": "failed", "reason": repr(exc)})
    return pd.DataFrame(rows), pd.DataFrame(failures)

pool_audit, pool_failures = diagnose_pool_builds(TEST_YEARS_TO_RUN)
display(pool_audit)
display(pool_failures)

pool_audit.to_csv(EXPERIMENT_OUTPUT_DIR / "ff6_pool_audit.csv", index=False, encoding="utf-8-sig")
pool_failures.to_csv(EXPERIMENT_OUTPUT_DIR / "ff6_pool_failures.csv", index=False, encoding="utf-8-sig")

,test_year,selection_date,n_train_months,n_test_months,n_raw_test_months,dropped_empty_test_months,n_stocks,n_action_stocks,n_features,n_dropped_features,train_mask_min,train_mask_max,test_mask_min,test_mask_max,eligible_stocks,ff6_selected_counts,ff6_action_counts,status
0,2018,2018-01-31,120,12,12,0,180,18,82,1,180,180,179,180,1496,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
1,2019,2019-01-31,120,12,12,0,180,18,83,0,180,180,177,180,1567,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
2,2020,2020-01-23,120,12,12,0,180,18,82,1,180,180,180,180,1654,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
3,2021,2021-01-29,120,12,12,0,180,18,83,0,180,180,178,180,1992,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
4,2022,2022-01-28,120,12,12,0,180,18,83,0,180,180,177,180,2251,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
5,2023,2023-01-31,120,12,12,0,180,18,83,0,180,180,177,180,2366,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
6,2024,2024-01-31,120,12,12,0,180,18,83,0,180,180,173,180,2328,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
7,2025,2025-01-27,120,12,12,0,180,18,83,0,180,180,179,180,2403,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok
8,2026,2026-01-30,120,4,5,1,180,18,83,0,180,180,180,180,2592,"{'Big_Growth': 30, 'Big_Neutral': 30, 'Big_Val...","{'Big_Growth': 3, 'Big_Neutral': 3, 'Big_Value...",ok


,test_year,status,reason
0,2017,failed,ValueError('2017: no stocks satisfy strict 120...


In [5]:
def make_env_for_data(data, factors, indices, reward_mode="log_return", action_mode="long_short", long_only_temperature=0.2):
    return FixedActionPortfolioEnv(
        factors=factors,
        forward_returns=data.arrays.ret_forward,
        mask=data.arrays.mask,
        dates=data.arrays.dates,
        indices=indices,
        action_slot_indices=data.action_slot_indices,
        lookback=LOOKBACK,
        reward_mode=reward_mode,
        action_mode=action_mode,
        long_only_temperature=long_only_temperature,
    )


def summarize_by_experiment(returns):
    frames = []
    for experiment, group in returns.groupby("experiment"):
        summary = performance_summary(group, method_col="method")
        summary.insert(0, "experiment", experiment)
        frames.append(summary)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def save_split_metadata(exp_name, data):
    prefix = EXPERIMENT_OUTPUT_DIR / f"{exp_name}_test{data.split.test_year}"
    data.stock_groups.to_csv(prefix.with_name(prefix.name + "_stock_groups.csv"), index=False, encoding="utf-8-sig")
    data.action_groups.to_csv(prefix.with_name(prefix.name + "_action_groups.csv"), index=False, encoding="utf-8-sig")
    data.missingness_report.to_csv(prefix.with_name(prefix.name + "_missingness.csv"), index=False, encoding="utf-8-sig")
    def _jsonify(obj):
        if isinstance(obj, dict):
            return {str(k) if not isinstance(k, (str, int, float, bool, type(None))) else k: _jsonify(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [_jsonify(v) for v in obj]
        return obj
    with open(prefix.with_name(prefix.name + "_audit.json"), "w", encoding="utf-8") as f:
        json.dump(_jsonify(data.audit), f, ensure_ascii=False, indent=2, default=str)


def run_one_experiment(config, test_years=None, total_timesteps=PPO_TOTAL_TIMESTEPS):
    exp_name = config["name"]
    action_mode = config.get("action_mode", "long_short")
    cafpo_method = config.get("cafpo_method", "CAFPO_PPO_LogReturn_5SeedAvgAction")
    include_markowitz = config.get("include_markowitz", True)
    long_only_temperature = config.get("long_only_temperature", LONG_ONLY_TEMPERATURE)
    if test_years is None:
        test_years = rolling_test_years(pre)

    all_returns = []
    all_histories = []
    skipped = []

    for year in test_years:
        print(f"[{exp_name}] build split test_year={year}")
        try:
            data = build_ff6_split_data(
                pre,
                manifest,
                test_year=year,
                n_stocks=N_STOCKS,
                n_action_stocks=config["n_action_stocks"],
                drop_sparse_features=config["drop_sparse_features"],
                seed=42,
            )
        except Exception as exc:
            print(f"[{exp_name}] skip test_year={year}: {exc}")
            skipped.append({"experiment": exp_name, "test_year": year, "reason": repr(exc)})
            continue

        save_split_metadata(exp_name, data)
        assert data.arrays.x.shape[1] == N_STOCKS
        assert data.arrays.x.shape[2] == len(data.kept_feature_cols)
        assert data.arrays.mask[data.split.train_idx].sum(axis=1).min() == N_STOCKS
        assert len(data.action_slot_indices) == config["n_action_stocks"]

        val_idx = data.split.train_idx[-12:]
        fit_idx = data.split.train_idx[:-12]
        cae_model, cae_history = train_cae(
            data.arrays,
            train_idx=fit_idx,
            val_idx=val_idx,
            n_factors=N_FACTORS,
            epochs=CAE_EPOCHS,
            patience=CAE_PATIENCE,
            seed=42 + int(year),
        )
        cae_history.insert(0, "experiment", exp_name)
        cae_history.insert(1, "test_year", year)
        all_histories.append(cae_history)

        factors, betas, reconstructed = extract_cae_outputs(cae_model, data.arrays)
        assert factors.shape == (len(data.arrays.dates), N_FACTORS)
        save_cae_outputs(
            factors,
            betas,
            reconstructed,
            data.arrays,
            EXPERIMENT_OUTPUT_DIR / f"{exp_name}_cae_outputs_test_{year}",
        )

        obs, _ = make_env_for_data(data, factors, data.split.train_idx[LOOKBACK:], action_mode=action_mode, long_only_temperature=long_only_temperature).reset()
        assert obs.shape == (LOOKBACK, N_FACTORS)
        assert make_env_for_data(data, factors, data.split.test_idx, action_mode=action_mode, long_only_temperature=long_only_temperature).action_space.shape == (config["n_action_stocks"],)

        models = []
        for seed in DRL_SEEDS:
            train_env = make_env_for_data(data, factors, data.split.train_idx[LOOKBACK:], action_mode=action_mode, long_only_temperature=long_only_temperature)
            model = make_ppo_model(
                train_env,
                seed=seed + int(year),
                n_steps=PPO_N_STEPS,
                batch_size=PPO_BATCH_SIZE,
                verbose=0,
            )
            model.learn(total_timesteps=total_timesteps, progress_bar=False)
            models.append(model)

        test_env = make_env_for_data(data, factors, data.split.test_idx, action_mode=action_mode, long_only_temperature=long_only_temperature)
        ppo = evaluate_raw_action_ensemble(models, test_env)
        ppo.insert(0, "experiment", exp_name)
        ppo.insert(1, "method", cafpo_method)
        ppo.insert(2, "test_year", year)

        action_arrays = subset_arrays(data.arrays, data.action_slot_indices)
        equal_weight = run_universe_weight_baseline(
            "EqualWeight",
            action_arrays,
            test_idx=data.split.test_idx,
            value_weighted=False,
        )
        value_weight = run_universe_weight_baseline(
            "ValueWeight",
            action_arrays,
            test_idx=data.split.test_idx,
            value_weighted=True,
        )
        frames = [equal_weight, value_weight, ppo]
        if include_markowitz:
            markowitz = run_markowitz_baseline(action_arrays, train_idx=data.split.train_idx, test_idx=data.split.test_idx)
            if action_mode == "long_only":
                markowitz["method"] = "Markowitz_LS"
            frames.append(markowitz)
        for frame in frames:
            if "experiment" not in frame.columns:
                frame.insert(0, "experiment", exp_name)
            if "test_year" not in frame.columns:
                frame.insert(frame.columns.get_loc("experiment") + 1 if "experiment" in frame.columns else 2, "test_year", year)

        all_returns.extend(frames)

    returns = pd.concat(all_returns, ignore_index=True) if all_returns else pd.DataFrame()
    histories = pd.concat(all_histories, ignore_index=True) if all_histories else pd.DataFrame()
    skipped_df = pd.DataFrame(skipped)
    summary = summarize_by_experiment(returns) if len(returns) else pd.DataFrame()

    returns.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_returns.csv", index=False, encoding="utf-8-sig")
    histories.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_cae_training_history.csv", index=False, encoding="utf-8-sig")
    skipped_df.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_skipped.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_summary.csv", index=False, encoding="utf-8-sig")
    return returns, histories, summary, skipped_df

In [6]:
## 5. Run Experiments

import importlib
import cafpo_reproduction as cr
import cafpo_ff6_experiments as fe
importlib.reload(cr)
importlib.reload(fe)
# re-bind symbols that were imported from these modules at the top of the notebook
from cafpo_reproduction import (
    OUTPUT_DIR,
    extract_cae_outputs,
    make_ppo_model,
    performance_summary,
    run_historical_sort_baseline,
    run_markowitz_baseline,
    save_cae_outputs,
    set_seed,
    train_cae,
)
from cafpo_ff6_experiments import (
    FF6_GROUP_ORDER,
    FixedActionPortfolioEnv,
    build_ff6_split_data,
    evaluate_raw_action_ensemble,
    load_prepanel_and_manifest,
    rolling_test_years,
    subset_arrays,
)
# Also reload the Plan A group-action module so the helper signatures stay in sync
# with any subsequent edits.
import cafpo_ff6_group_action as fga
importlib.reload(fga)
from cafpo_ff6_group_action import (
    FF6GroupPortfolioEnv,
    build_ff6_group_returns,
    ff6_group_diagnostics,
    run_ff6_group_weight_baseline,
)
set_seed(42)
print("modules reloaded")

modules reloaded


In [7]:
PLAN_A_N_STOCKS = 1056
PLAN_A_N_GROUPS = len(FF6_GROUP_ORDER)
PLAN_A_DROP_SPARSE_FEATURES = True
PLAN_A_LONG_ONLY_TEMPERATURE = 0.2
# 2026-07-07: T=0.2 only for the external-CAE Plan A run
PLAN_A_TEMPERATURES = [0.2]
PLAN_A_TOTAL_TIMESTEPS = PPO_TOTAL_TIMESTEPS
# Run all feasible full years. 2017 fails (no ret_1m in 2007-01), 2026 only has 4 test months, so restrict to 2018-2025.
PLAN_A_TEST_YEARS_TO_RUN = list(range(2018, 2026))
PLAN_A_OUTPUT_PREFIX = "expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly"
# Smoke knobs: tiny timesteps and a short year list for quick verification.
PLAN_A_SMOKE = False
PLAN_A_SMOKE_TIMESTEPS = 256
PLAN_A_SMOKE_YEARS = [2018]
# Set to True to launch the full Plan A training. CAE is trained once per test year
# and shared across all temperatures; the PPO loop is rerun per temperature.
RUN_PLAN_A_EXPERIMENT = True
from cafpo_ff6_group_action import (
    FF6GroupPortfolioEnv,
    build_ff6_group_returns,
    ff6_group_diagnostics,
    run_ff6_group_weight_baseline,
)
def _plan_a_method_label(temperature: float) -> str:
    tag = str(temperature).replace(".", "p")
    return f"CAFPO_PPO_LogReturn_FF6Group6_ValueWeighted_T{tag}_5SeedAvgAction"
def _plan_a_exp_name(temperature: float) -> str:
    tag = str(temperature).replace(".", "p")
    return f"{PLAN_A_OUTPUT_PREFIX}_T{tag}"
def make_plan_a_env(factors, group_returns, dates, indices, reward_mode="log_return", long_only_temperature=PLAN_A_LONG_ONLY_TEMPERATURE):
    return FF6GroupPortfolioEnv(
        factors=factors,
        group_returns=group_returns,
        dates=dates,
        indices=indices,
        lookback=LOOKBACK,
        reward_mode=reward_mode,
        long_only_temperature=long_only_temperature,
    )
def run_plan_a_experiment(test_years=None, total_timesteps=PLAN_A_TOTAL_TIMESTEPS, temperatures=None):
    temperatures = list(PLAN_A_TEMPERATURES if temperatures is None else temperatures)
    assert len(temperatures) >= 1, "temperatures must be a non-empty list"
    if test_years is None:
        test_years = rolling_test_years(pre)
    all_returns = []
    all_histories = []
    all_group_diagnostics = []
    skipped = []
    per_temp_returns = {t: [] for t in temperatures}
    per_temp_summaries = {}
    for year in test_years:
        print(f"[{PLAN_A_OUTPUT_PREFIX}] build split test_year={year}")
        try:
            data = build_ff6_split_data(
                pre,
                manifest,
                test_year=year,
                n_stocks=PLAN_A_N_STOCKS,
                n_action_stocks=PLAN_A_N_STOCKS,
                drop_sparse_features=PLAN_A_DROP_SPARSE_FEATURES,
                seed=42,
            )
        except Exception as exc:
            print(f"[{PLAN_A_OUTPUT_PREFIX}] skip test_year={year}: {exc}")
            skipped.append({"experiment": PLAN_A_OUTPUT_PREFIX, "test_year": year, "reason": repr(exc)})
            continue
        data.audit["plan_a_action_dim"] = PLAN_A_N_GROUPS
        data.audit["plan_a_group_weighting"] = "monthly value-weighted within each FF6 group"
        data.audit["plan_a_temperatures"] = temperatures
        save_split_metadata(PLAN_A_OUTPUT_PREFIX, data)
        assert data.arrays.x.shape[1] == PLAN_A_N_STOCKS
        assert data.arrays.x.shape[2] == len(data.kept_feature_cols)
        assert data.arrays.mask[data.split.train_idx].sum(axis=1).min() == PLAN_A_N_STOCKS
        val_idx = data.split.train_idx[-12:]
        fit_idx = data.split.train_idx[:-12]
        cae_model, cae_history = train_cae(
            data.arrays,
            train_idx=fit_idx,
            val_idx=val_idx,
            n_factors=N_FACTORS,
            epochs=CAE_EPOCHS,
            patience=CAE_PATIENCE,
            seed=42 + int(year),
        )
        if "experiment" not in cae_history.columns:
            cae_history.insert(0, "experiment", PLAN_A_OUTPUT_PREFIX)
        if "test_year" not in cae_history.columns:
            cae_history.insert(1, "test_year", year)
        all_histories.append(cae_history)
        factors, betas, reconstructed = extract_cae_outputs(cae_model, data.arrays)
        assert factors.shape == (len(data.arrays.dates), N_FACTORS)
        save_cae_outputs(
            factors,
            betas,
            reconstructed,
            data.arrays,
            EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_cae_outputs_test_{year}",
        )
        group_returns = build_ff6_group_returns(data)
        group_diag = ff6_group_diagnostics(group_returns, data.arrays.dates)
        group_diag.insert(0, "experiment", PLAN_A_OUTPUT_PREFIX)
        group_diag.insert(1, "test_year", year)
        all_group_diagnostics.append(group_diag)
        group_diag.to_csv(
            EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_test{year}_ff6_group_returns.csv",
            index=False,
            encoding="utf-8-sig",
        )
        train_indices = data.split.train_idx[LOOKBACK:]
        # Baselines: shared across temperatures
        group_equal = run_ff6_group_weight_baseline(
            "FF6GroupEqualWeight",
            group_returns,
            dates=data.arrays.dates,
            test_idx=data.split.test_idx,
            value_weighted=False,
        )
        group_value = run_ff6_group_weight_baseline(
            "FF6GroupValueWeight",
            group_returns,
            dates=data.arrays.dates,
            test_idx=data.split.test_idx,
            value_weighted=True,
        )
        for frame in (group_equal, group_value):
            frame.insert(0, "experiment", PLAN_A_OUTPUT_PREFIX)
            frame.insert(1, "test_year", year)
        all_returns.extend([group_equal, group_value])
        # PPO loop per temperature (CAE factors are shared)
        for temperature in temperatures:
            print(f"[{PLAN_A_OUTPUT_PREFIX}] train PPO T={temperature} test_year={year}")
            models = []
            for seed in DRL_SEEDS:
                train_env = make_plan_a_env(
                    factors=factors,
                    group_returns=group_returns,
                    dates=data.arrays.dates,
                    indices=train_indices,
                    long_only_temperature=temperature,
                )
                model = make_ppo_model(
                    train_env,
                    seed=seed + int(year),
                    n_steps=PPO_N_STEPS,
                    batch_size=PPO_BATCH_SIZE,
                    verbose=0,
                )
                model.learn(total_timesteps=total_timesteps, progress_bar=False)
                models.append(model)
            test_env = make_plan_a_env(
                factors=factors,
                group_returns=group_returns,
                dates=data.arrays.dates,
                indices=data.split.test_idx,
                long_only_temperature=temperature,
            )
            ppo = evaluate_raw_action_ensemble(models, test_env)
            ppo.insert(0, "experiment", _plan_a_exp_name(temperature))
            ppo.insert(1, "method", _plan_a_method_label(temperature))
            ppo.insert(2, "test_year", year)
            ppo["temperature"] = temperature
            per_temp_returns[temperature].append(ppo)
    returns = pd.concat(all_returns, ignore_index=True) if all_returns else pd.DataFrame()
    histories = pd.concat(all_histories, ignore_index=True) if all_histories else pd.DataFrame()
    group_diagnostics = pd.concat(all_group_diagnostics, ignore_index=True) if all_group_diagnostics else pd.DataFrame()
    skipped_df = pd.DataFrame(skipped)
    for temperature, frames in per_temp_returns.items():
        temp_returns = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        per_temp_summaries[temperature] = summarize_by_experiment(temp_returns) if len(temp_returns) else pd.DataFrame()
        temp_returns.to_csv(
            EXPERIMENT_OUTPUT_DIR / f"{_plan_a_exp_name(temperature)}_returns.csv",
            index=False,
            encoding="utf-8-sig",
        )
    # Combined PPO returns across all temperatures for the joint summary
    all_ppo_frames = [df for df in per_temp_returns.values() if isinstance(df, pd.DataFrame) and len(df)]
    all_ppo_returns = (
        pd.concat(all_ppo_frames, ignore_index=True)
        if all_ppo_frames
        else pd.DataFrame()
    )
    combined_returns = (
        pd.concat([returns, all_ppo_returns], ignore_index=True)
        if len(returns) and len(all_ppo_returns)
        else (returns if len(returns) else all_ppo_returns)
    )
    summary = summarize_by_experiment(combined_returns) if len(combined_returns) else pd.DataFrame()
    histories.to_csv(EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_cae_training_history.csv", index=False, encoding="utf-8-sig")
    group_diagnostics.to_csv(EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_ff6_group_returns.csv", index=False, encoding="utf-8-sig")
    skipped_df.to_csv(EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_skipped.csv", index=False, encoding="utf-8-sig")
    combined_returns.to_csv(EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_returns.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_summary.csv", index=False, encoding="utf-8-sig")
    if per_temp_summaries:
        temp_compare = pd.concat(
            [df.assign(temperature=t) for t, df in per_temp_summaries.items() if len(df)],
            ignore_index=True,
        )
        temp_compare.to_csv(
            EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_OUTPUT_PREFIX}_temperature_compare.csv",
            index=False,
            encoding="utf-8-sig",
        )
    return combined_returns, histories, group_diagnostics, summary, skipped_df, per_temp_summaries
if RUN_PLAN_A_EXPERIMENT:
    plan_a_years = PLAN_A_TEST_YEARS_TO_RUN or rolling_test_years(pre)
    if PLAN_A_SMOKE:
        plan_a_years = [y for y in plan_a_years if y in set(PLAN_A_SMOKE_YEARS)] or PLAN_A_SMOKE_YEARS
        plan_a_timesteps = PLAN_A_SMOKE_TIMESTEPS
        print(f"[Plan A smoke] years={plan_a_years} timesteps={plan_a_timesteps}")
    else:
        plan_a_timesteps = PLAN_A_TOTAL_TIMESTEPS
    plan_a_returns, plan_a_histories, plan_a_group_diagnostics, plan_a_summary, plan_a_skipped, plan_a_temp_summaries = run_plan_a_experiment(
        test_years=plan_a_years,
        total_timesteps=plan_a_timesteps,
        temperatures=PLAN_A_TEMPERATURES,
    )
    display(plan_a_summary)
    display(plan_a_skipped)
    if plan_a_temp_summaries:
        temp_summary_df = pd.concat(
            [df.assign(temperature=t) for t, df in plan_a_temp_summaries.items() if len(df)],
            ignore_index=True,
        )
        display(temp_summary_df)
else:
    print("RUN_PLAN_A_EXPERIMENT is False. Set it to True to train Plan A.")
    print(f"Plan A uses {PLAN_A_N_STOCKS} stocks ({PLAN_A_N_STOCKS // PLAN_A_N_GROUPS} per FF6 group) and {PLAN_A_N_GROUPS} PPO actions.")
    print(f"Temperature sweep: {PLAN_A_TEMPERATURES}.")


[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] build split test_year=2018
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] train PPO T=0.2 test_year=2018


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] build split test_year=2019
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] train PPO T=0.2 test_year=2019


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] build split test_year=2020
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] train PPO T=0.2 test_year=2020


...

[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] build split test_year=2025
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly] train PPO T=0.2 test_year=2025


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


,experiment,method,months,compound_return,sharpe_ratio,max_drawdown,sterling_ratio
0,expA_ff6_fixed1056_featurefilter_group6_valuew...,FF6GroupEqualWeight,96,0.530564,0.107703,-0.263103,0.022353
1,expA_ff6_fixed1056_featurefilter_group6_valuew...,FF6GroupValueWeight,96,0.482848,0.110060,-0.235376,0.022021


""


,experiment,method,months,compound_return,sharpe_ratio,max_drawdown,sterling_ratio,temperature
0,expA_ff6_fixed1056_featurefilter_group6_valuew...,CAFPO_PPO_LogReturn_FF6Group6_ValueWeighted_T0...,96,0.503117,0.097821,-0.414865,0.014959,0.2


In [30]:
import re as _re
import math as _math
import pandas as _pd
import numpy as _np
from pathlib import Path as _Path

# Use the experiment output directory from cell 2 if present, else default to the on-disk location.
_PLAN_A_BASE = EXPERIMENT_OUTPUT_DIR if "EXPERIMENT_OUTPUT_DIR" in dir() else _Path(
    "c:/xmrs_workspace/workspace/【广发】深度学习play/cafpo/rqdata_output/ff6_fixed180_experiments"
)

def _compound(returns: _pd.Series) -> float:
    return float((1.0 + returns).prod() - 1.0)

def _max_drawdown(returns: _pd.Series) -> float:
    nav = (1.0 + returns).cumprod()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    return float(dd.min())

# Original rolling-window Plan A (T=0.2, 0.5, 1.0) - per-temperature files.
_rolling_files = sorted(_PLAN_A_BASE.glob(f"{PLAN_A_OUTPUT_PREFIX}_T*_returns.csv"))
# Fixed-start Plan A (T=0.2 only).
_fixed_files = sorted(_PLAN_A_BASE.glob(f"{PLAN_A_OUTPUT_PREFIX}_FixedTrainStartFrom2007_T*_returns.csv"))

def _load_ppo(files):
    if not files:
        return None
    frames = []
    for _f in files:
        _df = _pd.read_csv(_f)
        # Filter to PPO method rows only. The per-temperature rolling files
        # already contain only PPO, but the fixed-start file currently has
        # baselines + PPO mixed in.
        if "method" in _df.columns:
            _df = _df[_df["method"].astype(str).str.contains("PPO", na=False)].copy()
        if _df.empty:
            continue
        _m = _re.search(r"_T(\d+)p(\d+)_returns\.csv$", _f.name)
        if _m is None:
            continue
        _temp = float(f"{_m.group(1)}.{_m.group(2)}")
        _df["temperature"] = _temp
        if "FixedTrainStartFrom2007" in _f.name:
            _df["variant"] = "fixedStart2007"
        else:
            _df["variant"] = "10y_rolling"
        frames.append(_df)
    return _pd.concat(frames, ignore_index=True) if frames else None

_ppo_rolling = _load_ppo(_rolling_files)
_ppo_fixed = _load_ppo(_fixed_files)

if _ppo_rolling is not None or _ppo_fixed is not None:
    _ppo_all = _pd.concat([x for x in [_ppo_rolling, _ppo_fixed] if x is not None], ignore_index=True)
    print(f"Loaded {len(_ppo_rolling) if _ppo_rolling is not None else 0} rolling PPO rows + "
          f"{len(_ppo_fixed) if _ppo_fixed is not None else 0} fixed-start PPO rows; "
          f"combined rows={len(_ppo_all)}; variant x temperature combos={sorted(set(zip(_ppo_all['variant'], _ppo_all['temperature'])))}")

    print("\n=== Plan A: PPO variant x temperature head-to-head (overall) ===")
    _ppo_overall = (_ppo_all.groupby(["variant", "temperature"])
                    .agg(n=("portfolio_return", "size"),
                         mean=("portfolio_return", "mean"),
                         std=("portfolio_return", "std"),
                         compound=("portfolio_return", _compound))
                    .reset_index())
    _ppo_overall["ann_sharpe"] = _ppo_overall["mean"] / _ppo_overall["std"] * _math.sqrt(12)
    _ppo_overall["max_dd"] = [_max_dd(g["portfolio_return"]) for _, g in _ppo_all.groupby(["variant", "temperature"])]
    display(_ppo_overall.round(4))

    print("\n=== Plan A: PPO mean monthly return per test_year by variant x temperature ===")
    _ppo_year_pivot = (_ppo_all.groupby(["test_year", "variant", "temperature"])["portfolio_return"]
                       .mean().unstack(["variant", "temperature"]))
    display(_ppo_year_pivot.round(4))

    print("\n=== Plan A: PPO annualized Sharpe per test_year by variant x temperature ===")
    _ppo_year_stats = (_ppo_all.groupby(["variant", "temperature", "test_year"])["portfolio_return"]
                       .agg(["mean", "std"]).reset_index())
    _ppo_year_stats["ann_sharpe"] = _ppo_year_stats["mean"] / _ppo_year_stats["std"] * _math.sqrt(12)
    _ppo_sharpe_pivot = _ppo_year_stats.pivot(
        index="test_year", columns=["variant", "temperature"], values="ann_sharpe"
    )
    display(_ppo_sharpe_pivot.round(3))

    if _ppo_fixed is not None and _ppo_rolling is not None:
        _fixed_02 = _ppo_fixed[_ppo_fixed["temperature"] == 0.2].copy()
        _roll_02 = _ppo_rolling[_ppo_rolling["temperature"] == 0.2].copy()
        if not _fixed_02.empty and not _roll_02.empty:
            _both = _pd.concat([_fixed_02.assign(variant="fixedStart2007"),
                                _roll_02.assign(variant="10y_rolling")], ignore_index=True)
            _both["date"] = _pd.to_datetime(_both["date"])
            _both["test_year"] = _both["date"].dt.year
            _overlap = sorted(set(_both["test_year"].unique()))
            _overlap = [y for y in _overlap if y in [2020, 2021, 2022, 2023, 2024, 2025]]
            _ovl = _both[_both["test_year"].isin(_overlap)].copy()
            _ovl_grp = _ovl.groupby("variant")["portfolio_return"].agg(["mean", "std"])
            _ovl_grp["ann_sharpe"] = _ovl_grp["mean"] / _ovl_grp["std"] * _math.sqrt(12)
            _ovl_grp["compound"] = _ovl.groupby("variant")["portfolio_return"].apply(_compound)
            _ovl_grp["max_dd"] = _ovl.groupby("variant")["portfolio_return"].apply(_max_dd)
            print(f"\n=== Overlap 2020-2025 (T=0.2) ===")
            display(_ovl_grp.round(4))

            _per_year_mean = _ovl.groupby(["test_year", "variant"])["portfolio_return"].mean().unstack("variant")
            _diff = _per_year_mean["fixedStart2007"] - _per_year_mean["10y_rolling"]
            print(f"\n=== Per-year mean monthly return difference (fixed - rolling) ===")
            display(_per_year_mean.round(4))
            print(f"Diff: \n{_diff.round(4)}")
            print(f"Mean improvement: {_diff.mean():+.4f}  |  Median: {_diff.median():+.4f}  |  Wins: {(_diff > 0).sum()}/{len(_diff)} years")
else:
    print("No Plan A PPO files found.")

Loaded 288 rolling PPO rows + 72 fixed-start PPO rows; combined rows=360; variant x temperature combos=[('10y_rolling', 0.2), ('10y_rolling', 0.5), ('10y_rolling', 1.0), ('fixedStart2007', 0.2)]

=== Plan A: PPO variant x temperature head-to-head (overall) ===


,variant,temperature,n,mean,std,compound,ann_sharpe,max_dd
0,10y_rolling,0.2,96,0.0036,0.0638,0.1707,0.1963,-0.4713
1,10y_rolling,0.5,96,0.0043,0.0567,0.3058,0.2648,-0.3315
2,10y_rolling,1.0,96,0.0049,0.0546,0.3891,0.3086,-0.2653
3,fixedStart2007,0.2,72,0.0039,0.0575,0.1830,0.2371,-0.3258



=== Plan A: PPO mean monthly return per test_year by variant x temperature ===


variant     10y_rolling                 fixedStart2007
temperature         0.2     0.5     1.0            0.2
test_year                                             
2018            -0.0252 -0.0272 -0.0276            NaN
2019             0.0309  0.0270  0.0211            NaN
2020             0.0270  0.0156  0.0106         0.0190
2021            -0.0115 -0.0013  0.0043        -0.0063
2022             0.0006  0.0008  0.0039         0.0015
2023            -0.0333 -0.0226 -0.0172        -0.0202
2024             0.0179  0.0185  0.0183         0.0161
2025             0.0225  0.0237  0.0256         0.0136


=== Plan A: PPO annualized Sharpe per test_year by variant x temperature ===


...


=== Overlap 2020-2025 (T=0.2) ===


,mean,std,ann_sharpe,compound,max_dd
variant,,,,,
10y_rolling,0.0039,0.0648,0.2062,0.1412,-0.4713
fixedStart2007,0.0039,0.0575,0.2371,0.1830,-0.3258



=== Per-year mean monthly return difference (fixed - rolling) ===


variant,10y_rolling,fixedStart2007
test_year,,
2020,0.0270,0.0190
2021,-0.0115,-0.0063
2022,0.0006,0.0015
2023,-0.0333,-0.0202
2024,0.0179,0.0161
2025,0.0225,0.0136


Diff: 
test_year
2020   -0.0080
2021    0.0052
2022    0.0009
2023    0.0131
2024   -0.0018
2025   -0.0089
dtype: float64
Mean improvement: +0.0001  |  Median: -0.0004  |  Wins: 3/6 years


In [8]:
# Plan A fixed-start extension
# Goal: keep the FF6-balanced 1056 stock pool as Plan A, but anchor the training window to
# start from 2007-01 regardless of the test year. Compare with the original 10-year rolling
# window (T=0.2 only) to see whether a longer training history improves PPO performance.
#
# Outputs are saved under ff6_fixed180_experiments with the prefix
# `expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2`
# so they live next to the original Plan A artefacts.
PLAN_A_FIXED_TRAIN_START_YEAR = 2007
PLAN_A_FIXED_OUTPUT_PREFIX = (
    "expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007"
)
PLAN_A_FIXED_TEMPERATURE = 0.2
PLAN_A_FIXED_TEST_YEARS = list(range(2020, 2026))  # 2020-2025 inclusive (2026 incomplete)
# Self-contained: depend only on cell 3 (PPO_TOTAL_TIMESTEPS) instead of cell 7.
PLAN_A_FIXED_TOTAL_TIMESTEPS = PPO_TOTAL_TIMESTEPS
# 2007-01 has no ret_1m in this dataset, so the original 100% training-month coverage
# rule cannot be satisfied for any 13-18 year training window. Relax to 90% (each
# selected stock must have ret_1m / ret_fwd_1m / raw feature in at least 90% of the
# training months). 95% is enough for 5/6 years but 2024's Big_Growth cell ends up
# with only 175 eligible stocks (need 176), so we drop to 90% to cover 2024 too.
# This is the only behavioural change vs the original 10y Plan A.
PLAN_A_FIXED_MIN_TRAIN_MONTHS_RATIO = 0.90
# Set to True to launch the full fixed-start training loop. CAE is trained once per
# test year (shared across the lone temperature T=0.2); PPO loop runs 5 seeds.
RUN_PLAN_A_FIXED_EXPERIMENT = True
def _fixed_start_method_label(temperature: float = PLAN_A_FIXED_TEMPERATURE) -> str:
    tag = str(temperature).replace(".", "p")
    return f"CAFPO_PPO_LogReturn_FF6Group6_ValueWeighted_FixedStart2007_T{tag}_5SeedAvgAction"
def _fixed_start_exp_name(temperature: float = PLAN_A_FIXED_TEMPERATURE) -> str:
    tag = str(temperature).replace(".", "p")
    return f"{PLAN_A_FIXED_OUTPUT_PREFIX}_T{tag}"
def _fixed_start_train_years(test_year: int) -> int:
    return int(test_year) - PLAN_A_FIXED_TRAIN_START_YEAR
def run_plan_a_fixed_start_experiment(test_years=None, total_timesteps=PLAN_A_FIXED_TOTAL_TIMESTEPS, temperature=PLAN_A_FIXED_TEMPERATURE):
    if test_years is None:
        test_years = list(PLAN_A_FIXED_TEST_YEARS)
    exp_name = _fixed_start_exp_name(temperature)
    method_label = _fixed_start_method_label(temperature)
    all_returns = []
    all_histories = []
    all_group_diagnostics = []
    skipped = []
    for year in test_years:
        ty = _fixed_start_train_years(year)
        print(f"[{exp_name}] build split test_year={year} train_years={ty}")
        try:
            data = build_ff6_split_data(
                pre,
                manifest,
                test_year=year,
                n_stocks=PLAN_A_N_STOCKS,
                n_action_stocks=PLAN_A_N_STOCKS,
                drop_sparse_features=PLAN_A_DROP_SPARSE_FEATURES,
                train_years=ty,
                seed=42,
                min_train_months_ratio=PLAN_A_FIXED_MIN_TRAIN_MONTHS_RATIO,
            )
        except Exception as exc:
            print(f"[{exp_name}] skip test_year={year}: {exc}")
            skipped.append({"experiment": exp_name, "test_year": year, "reason": repr(exc)})
            continue
        data.audit["plan_a_action_dim"] = PLAN_A_N_GROUPS
        data.audit["plan_a_group_weighting"] = "monthly value-weighted within each FF6 group"
        data.audit["plan_a_fixed_train_start_year"] = PLAN_A_FIXED_TRAIN_START_YEAR
        data.audit["plan_a_fixed_train_years"] = ty
        data.audit["plan_a_fixed_min_train_months_ratio"] = PLAN_A_FIXED_MIN_TRAIN_MONTHS_RATIO
        save_split_metadata(PLAN_A_FIXED_OUTPUT_PREFIX, data)
        assert data.arrays.x.shape[1] == PLAN_A_N_STOCKS
        assert data.arrays.x.shape[2] == len(data.kept_feature_cols)
        # With min_train_months_ratio < 1.0 some training months will not have a
        # full 1056-stock mask; that's expected and the CAE/PPO handle it via
        # the per-stock mask channel. We only require the median training month
        # to retain a near-full mask, and that test months have at least 50%.
        _train_mask_median = int(pd.Series(data.arrays.mask[data.split.train_idx].sum(axis=1)).median())
        _test_mask_median = int(pd.Series(data.arrays.mask[data.split.test_idx].sum(axis=1)).median())
        assert _train_mask_median >= int(PLAN_A_N_STOCKS * 0.95), (
            f"train median mask {_train_mask_median} too low (want >= {int(PLAN_A_N_STOCKS * 0.95)})"
        )
        assert _test_mask_median >= int(PLAN_A_N_STOCKS * 0.50), (
            f"test median mask {_test_mask_median} too low (want >= {int(PLAN_A_N_STOCKS * 0.50)})"
        )
        val_idx = data.split.train_idx[-12:]
        fit_idx = data.split.train_idx[:-12]
        cae_model, cae_history = train_cae(
            data.arrays,
            train_idx=fit_idx,
            val_idx=val_idx,
            n_factors=N_FACTORS,
            epochs=CAE_EPOCHS,
            patience=CAE_PATIENCE,
            seed=42 + int(year),
        )
        if "experiment" not in cae_history.columns:
            cae_history.insert(0, "experiment", exp_name)
        if "test_year" not in cae_history.columns:
            cae_history.insert(1, "test_year", year)
        all_histories.append(cae_history)
        factors, betas, reconstructed = extract_cae_outputs(cae_model, data.arrays)
        assert factors.shape == (len(data.arrays.dates), N_FACTORS)
        save_cae_outputs(
            factors,
            betas,
            reconstructed,
            data.arrays,
            EXPERIMENT_OUTPUT_DIR / f"{exp_name}_cae_outputs_test_{year}",
        )
        group_returns = build_ff6_group_returns(data)
        group_diag = ff6_group_diagnostics(group_returns, data.arrays.dates)
        group_diag.insert(0, "experiment", exp_name)
        group_diag.insert(1, "test_year", year)
        all_group_diagnostics.append(group_diag)
        group_diag.to_csv(
            EXPERIMENT_OUTPUT_DIR / f"{exp_name}_test{year}_ff6_group_returns.csv",
            index=False,
            encoding="utf-8-sig",
        )
        train_indices = data.split.train_idx[LOOKBACK:]
        group_equal = run_ff6_group_weight_baseline(
            "FF6GroupEqualWeight",
            group_returns,
            dates=data.arrays.dates,
            test_idx=data.split.test_idx,
            value_weighted=False,
        )
        group_value = run_ff6_group_weight_baseline(
            "FF6GroupValueWeight",
            group_returns,
            dates=data.arrays.dates,
            test_idx=data.split.test_idx,
            value_weighted=True,
        )
        for frame in (group_equal, group_value):
            frame.insert(0, "experiment", exp_name)
            frame.insert(1, "test_year", year)
        all_returns.extend([group_equal, group_value])
        print(f"[{exp_name}] train PPO T={temperature} test_year={year}")
        models = []
        for seed in DRL_SEEDS:
            train_env = make_plan_a_env(
                factors=factors,
                group_returns=group_returns,
                dates=data.arrays.dates,
                indices=train_indices,
                long_only_temperature=temperature,
            )
            model = make_ppo_model(
                train_env,
                seed=seed + int(year),
                n_steps=PPO_N_STEPS,
                batch_size=PPO_BATCH_SIZE,
                verbose=0,
            )
            model.learn(total_timesteps=total_timesteps, progress_bar=False)
            models.append(model)
        test_env = make_plan_a_env(
            factors=factors,
            group_returns=group_returns,
            dates=data.arrays.dates,
            indices=data.split.test_idx,
            long_only_temperature=temperature,
        )
        ppo = evaluate_raw_action_ensemble(models, test_env)
        ppo.insert(0, "experiment", exp_name)
        ppo.insert(1, "method", method_label)
        ppo.insert(2, "test_year", year)
        ppo["temperature"] = temperature
        all_returns.append(ppo)
        # Persist partial results after each year so a crash doesn't lose everything.
        # Concatenate whatever is collected so far (different methods can have slightly
        # different columns; pd.concat with sort=True handles the union).
        _partial = pd.concat(all_returns, ignore_index=True) if all_returns else pd.DataFrame()
        _partial.to_csv(
            EXPERIMENT_OUTPUT_DIR / f"{exp_name}_returns.csv", index=False, encoding="utf-8-sig"
        )
        _done = sorted({int(r['test_year'].iloc[0]) for r in all_returns if 'test_year' in r.columns})
        print(f"[{exp_name}] partial results saved: {len(_partial)} rows, test_years_done={_done}", flush=True)
    returns = pd.concat(all_returns, ignore_index=True) if all_returns else pd.DataFrame()
    histories = pd.concat(all_histories, ignore_index=True) if all_histories else pd.DataFrame()
    group_diagnostics = pd.concat(all_group_diagnostics, ignore_index=True) if all_group_diagnostics else pd.DataFrame()
    skipped_df = pd.DataFrame(skipped)
    summary = summarize_by_experiment(returns) if len(returns) else pd.DataFrame()
    returns.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_returns.csv", index=False, encoding="utf-8-sig")
    histories.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_cae_training_history.csv", index=False, encoding="utf-8-sig")
    group_diagnostics.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_ff6_group_returns.csv", index=False, encoding="utf-8-sig")
    skipped_df.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_skipped.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(EXPERIMENT_OUTPUT_DIR / f"{exp_name}_summary.csv", index=False, encoding="utf-8-sig")
    return returns, histories, group_diagnostics, summary, skipped_df
if RUN_PLAN_A_FIXED_EXPERIMENT:
    fixed_returns, fixed_histories, fixed_group_diagnostics, fixed_summary, fixed_skipped = run_plan_a_fixed_start_experiment(
        test_years=PLAN_A_FIXED_TEST_YEARS,
        total_timesteps=PLAN_A_FIXED_TOTAL_TIMESTEPS,
        temperature=PLAN_A_FIXED_TEMPERATURE,
    )
    display(fixed_summary)
    display(fixed_skipped)
else:
    print("RUN_PLAN_A_FIXED_EXPERIMENT is False. Set it to True to train fixed-start Plan A.")
# --- Head-to-head: Plan A 10y rolling (T=0.2) vs Plan A fixed-start-from-2007 (T=0.2) ---
# Both runs use the same 1056 FF6-balanced pool, the same CAE architecture, the same
# long-only softmax temperature, and the same 5 PPO seeds. The only difference is the
# training window: rolling uses test_year-10..test_year-1, fixed-start uses
# 2007-01..test_year-1.
# This comparison is only meaningful for the overlapping years 2020-2025.
from pathlib import Path as _Path2
_orig_ppo_path = _Path2(EXPERIMENT_OUTPUT_DIR) / f"{PLAN_A_OUTPUT_PREFIX}_T0p2_returns.csv"
_fixed_ppo_path = _Path2(EXPERIMENT_OUTPUT_DIR) / f"{PLAN_A_FIXED_OUTPUT_PREFIX}_T0p2_returns.csv"
if _orig_ppo_path.exists() and _fixed_ppo_path.exists():
    _orig = pd.read_csv(_orig_ppo_path)
    _fixed = pd.read_csv(_fixed_ppo_path)
    _orig = _orig[_orig["method"].str.contains("PPO", na=False)].copy()
    _fixed = _fixed[_fixed["method"].str.contains("PPO", na=False)].copy()
    _orig["variant"] = "PlanA_10y_rolling_T0p2"
    _fixed["variant"] = "PlanA_fixedStart2007_T0p2"
    _cmp = pd.concat([_orig, _fixed], ignore_index=True)
    _cmp["date"] = pd.to_datetime(_cmp["date"])
    _cmp["test_year"] = _cmp["date"].dt.year
    _overlap = sorted(set(_cmp["test_year"].unique()))
    _cmp_overlap = _cmp[_cmp["test_year"].isin(PLAN_A_FIXED_TEST_YEARS)].copy()
    print(f"Loaded PPO returns: orig={len(_orig)} rows, fixed={len(_fixed)} rows; overlap years={_overlap}")
    print(f"Head-to-head restricted to fixed-start years 2020-2025: {len(_cmp_overlap)} rows")
    def _compound(s):
        return float((1.0 + s).prod() - 1.0)
    def _max_dd(s):
        nav = (1.0 + s).cumprod()
        peak = nav.cummax()
        return float((nav / peak - 1.0).min())
    _per_variant = (_cmp_overlap.groupby("variant")
                    .agg(n=("portfolio_return", "size"),
                         mean=("portfolio_return", "mean"),
                         std=("portfolio_return", "std"),
                         compound=("portfolio_return", _compound))
                    .reset_index())
    _per_variant["ann_sharpe"] = _per_variant["mean"] / _per_variant["std"] * np.sqrt(12)
    _per_variant["max_dd"] = [_max_dd(_cmp_overlap.loc[_cmp_overlap["variant"] == v, "portfolio_return"])
                              for v in _per_variant["variant"]]
    _per_variant.to_csv(EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_FIXED_OUTPUT_PREFIX}_vs_10y_overall.csv",
                        index=False, encoding="utf-8-sig")
    print("\n=== Overall (overlap 2020-2025, 72 months) ===")
    display(_per_variant.round(4))
    _per_year = (_cmp_overlap.groupby(["variant", "test_year"])["portfolio_return"]
                 .agg(["mean", "std"]).reset_index())
    _per_year["ann_sharpe"] = _per_year["mean"] / _per_year["std"] * np.sqrt(12)
    _per_year_pivot = _per_year.pivot(index="test_year", columns="variant", values="ann_sharpe")
    _per_year_pivot.to_csv(EXPERIMENT_OUTPUT_DIR / f"{PLAN_A_FIXED_OUTPUT_PREFIX}_vs_10y_sharpe_by_year.csv",
                            encoding="utf-8-sig")
    print("\n=== Annualized Sharpe per test_year (overlap 2020-2025) ===")
    display(_per_year_pivot.round(3))
    _mean_year_pivot = (_cmp_overlap.groupby(["test_year", "variant"])["portfolio_return"]
                        .mean().unstack("variant"))
    print("\n=== Mean monthly return per test_year (overlap 2020-2025) ===")
    display(_mean_year_pivot.round(4))
    _diff = _mean_year_pivot["PlanA_fixedStart2007_T0p2"] - _mean_year_pivot["PlanA_10y_rolling_T0p2"]
    print("\n=== Improvement (fixed - rolling) per year ===")
    display(_diff.round(4))
    print(f"Mean improvement: {_diff.mean():+.4f}  |  Median improvement: {_diff.median():+.4f}  |  Wins: {(_diff > 0).sum()}/{len(_diff)} years")
else:
    print(f"Missing files for head-to-head: orig={_orig_ppo_path.exists()}  fixed={_fixed_ppo_path.exists()}")


[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] build split test_year=2020 train_years=13
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] train PPO T=0.2 test_year=2020


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] partial results saved: 36 rows, test_years_done=[2020]
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] build split test_year=2021 train_years=14
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] train PPO T=0.2 test_year=2021


c:\Users\ianning\AppData\Local\drl_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] partial results saved: 72 rows, test_years_done=[2020, 2021]
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] build split test_year=2022 train_years=15
[expA_ff6_fixed1056_featurefilter_group6_valueweighted_longonly_FixedTrainStartFrom2007_T0p2] train PPO T=0.2 test_year=2022


...


=== Mean monthly return per test_year (overlap 2020-2025) ===


variant,PlanA_10y_rolling_T0p2,PlanA_fixedStart2007_T0p2
test_year,,
2020,0.0344,0.0342
2021,-0.0098,-0.0080
2022,-0.0004,0.0022
2023,-0.0256,-0.0265
2024,0.0179,0.0209
2025,0.0242,0.0191



=== Improvement (fixed - rolling) per year ===


test_year
2020   -0.0002
2021    0.0018
2022    0.0026
2023   -0.0009
2024    0.0030
2025   -0.0051
dtype: float64

Mean improvement: +0.0002  |  Median improvement: +0.0008  |  Wins: 3/6 years
